In [48]:
!pip install ~/fwiVis/utility_functions/
! pip install plotnine
! pip install pymc


Processing /home/jovyan/fwiVis/utility_functions
  Preparing metadata (setup.py) ... done
  Created wheel for fwiVis: filename=fwiVis-0.1-py3-none-any.whl size=17985 sha256=b83c3b619739b60ba04477a04b0a487c1f01573159d9d2bf66ae10bc775a05c5
  Stored in directory: /tmp/pip-ephem-wheel-cache-mubo3val/wheels/79/0f/3d/08c18473dd7e0fb915900e6b4f13b81f1fa84371f9bf24d864
Successfully built fwiVis
  Attempting uninstall: fwiVis
    Found existing installation: fwiVis 0.1
    Uninstalling fwiVis-0.1:
      Successfully uninstalled fwiVis-0.1


In [49]:
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain

from datetime import date
from bs4 import BeautifulSoup
import requests
import os
import plotnine
import xarray as xr

#import numpy as np
from matplotlib import pyplot as plt
from plotnine import ggplot, geom_point, geom_jitter, aes, stat_smooth, facet_wrap
import plotnine as plotnine
import seaborn as sns

from scipy import stats
from patsy import ModelDesc
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

import math


In [53]:
import fwiVis.fwiVis as fv

#path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_only/April_1_unmerged_fires_with_FWI.csv"
#path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Final_dataset_as_of_20240209.csv"
#fire3 = fv.prep_fire_files(path)

#path = os.path.abspath("data/Quebec_v3_full_data_perimeters20241112.csv")
path = os.path.abspath("data/Quebec_v3_full_data_perimetersMAX_fwi_with_ISI_and_BUIful_timeseries_202534.csv")
fire3 = fv.prep_fire_files(path)
fire_first = fire3




In [54]:
fire3 = fire3.sort_values(by = ["fireID", "t"])

fire3 = fire3[~fire3.FWI.isna()]
# df = fire3[fire3.fireID == '2367']
# df


In [55]:
def corrected_farea_diff(df, var = "farea"):
    #sub_df = df[]
    df = df.drop_duplicates()
    df[f"{var}_diff"] = df[var].diff()
    min_t = df.t.min()
    #print()
    #print(df.loc[df.t == min_t, ["fireID", "t", "farea", "farea_diff"]])
    
    if(np.isnan(*df.loc[df.t == min_t, [f"{var}_diff"]].values)):
        df.loc[df.t == min_t, [f"{var}_diff"]] = df.loc[df.t == min_t, [f"{var}"]].values
    else:
        print("Error! First value not NaN")
        df = None
    return(df)

In [56]:
def just_the_igs(df,): ## Reminder that this will only acocunt for the earliest ignition from a multi-fire complex. 

    df["is_ig"] = False
    df.loc[df.t == df.t.min(), ['is_ig']] = True
    
    return(df)

def extinction(df, days_past = 1):
    df.loc[:, ['is_ext']] = False
    df.loc[:, ["t"]] = df.t.astype("datetime64[ns]")
    final_t = df[df.n_newpixels > 0].t.astype("datetime64[ns]").max()
    post_fire = final_t + timedelta(days = days_past)

    if (pd.isnull(np.datetime64(str(final_t)))):
        df.loc[(df.t == df.t.min()), ['is_ext']] = True
    else:
        df.loc[(df.t  > final_t)  & (df.t  <= post_fire), ['is_ext']] = True
    #print(final_t)
    #print(df.fireID.unique())
    #print(len(df[df["is_ext"] == True]))
    return(df)


fire3 = fire3[~fire3.FWI.isna()]





In [57]:
def chop_fires_at_end(df):
    final_t = df[df.n_newpixels > 0].t.max()
    df = df[df.t <= final_t]
    return(df)

#post_fr = fire3.groupby("fireID").apply(extinction).reset_index(drop = True)
fire3 = fire3.groupby("fireID").apply(chop_fires_at_end).reset_index(drop = True)


fire3 = fire3.sort_values(by = ["fireID", "t"])
fire3 = fire3[~fire3.FWI.isna()]

/tmp/ipykernel_489/455317430.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [58]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [59]:
### make a plot of the distribution of forwar-facing relative fire area growth
fire3 = fire3.drop_duplicates()
fire3["farea_diff_stand"] = fire3.groupby("fireID").farea.diff()

fire3["corrected_flinelen"] = fire3.flinelen + 0.001

## Fireline 
fire3["frac_change"] = fire3.groupby("fireID").farea.pct_change()
fire3["flinelen_frac_change"] = fire3.groupby("fireID").corrected_flinelen.pct_change()

fire3["fperim_shifted"] = fire3.groupby("fireID").fperim.shift(periods = 1)
fire3["flinelen_diff"] = fire3.groupby("fireID").flinelen.diff()
fire3["fperim_diff"] = fire3.groupby("fireID").fperim.diff()
fire3["farea_shifted_back"] = fire3.groupby("fireID").farea.shift(periods = 1)


fire3["farea_shifted"] = fire3.groupby("fireID").farea.shift(periods = 1)
fire3["meanFRP_shifted"] = fire3.groupby("fireID").meanFRP.shift(periods = 1)
fire3["flinelen_shifted"] = fire3.groupby("fireID").flinelen.shift(periods = 1)
fire3["n_newpixels_shifted"] = fire3.groupby("fireID").n_newpixels.shift(periods = 1)
fire3["GEOS5_IMERGEARLY_shifted"] = fire3.groupby("fireID")['GEOS-5.IMERGEARLY'].shift(periods = 1)
fire3["FWI_shifted"] = fire3.groupby("fireID")['FWI'].shift(periods = 1)
fire3["FWI_lead_1_shifted"] = fire3.groupby("fireID")['FWI_lead_1'].shift(periods = 1)
fire3["FWI_forecast_average"] = (fire3.FWI + fire3.FWI_lead_1)/2
fire3["FWI_forecast_average_shifted"] = (fire3.FWI_shifted + fire3.FWI_lead_1_shifted)/2
fire3["ISI_shifted"] = fire3.groupby("fireID")['ISI'].shift(periods = 1)
fire3["ISI_lead_1_shifted"] = fire3.groupby("fireID")['ISI_lead_1'].shift(periods = 1)
fire3["ISI_forecast_average"] = (fire3.ISI + fire3.ISI_lead_1)/2
fire3["ISI_forecast_average_shifted"] = (fire3.ISI_shifted + fire3.ISI_lead_1_shifted)/2
fire3["BUI_shifted"] = fire3.groupby("fireID")['BUI'].shift(periods = 1)
fire3["BUI_lead_1_shifted"] = fire3.groupby("fireID")['BUI_lead_1'].shift(periods = 1)
fire3["BUI_forecast_average"] = (fire3.BUI + fire3.BUI_lead_1)/2
fire3["BUI_forecast_average_shifted"] = (fire3.BUI_shifted + fire3.BUI_lead_1_shifted)/2




fire3["corrected_shifted_flinelen"] = fire3.flinelen_shifted + 0.001
fire3["flinelen_shifted_frac_change"] = fire3.groupby("fireID").corrected_shifted_flinelen.pct_change()

fire3["farea_shifted_exp"] = fire3["farea_shifted"] ** (1/10)
fire3["meanFRP_shifted_exp"] = fire3["meanFRP_shifted"]** (1/10)
fire3["flinelen_shifted_exp"] = fire3["flinelen_shifted"]** (1/10)
fire3["n_newpixels_shifted_exp"] = fire3["n_newpixels_shifted"]** (1/10)


fire3["spread_bool"] = fire3.frac_change > 0
fire3["spread_bool"] = fire3["spread_bool"].astype("int")


fire3["spread_bool_50"] = fire3.frac_change > 0.5
fire3["spread_bool_50"] = fire3["spread_bool_50"].astype("int")


fire3["spread_bool_100"] = fire3.frac_change > 1
fire3["spread_bool_100"] = fire3["spread_bool_100"].astype("int")

/tmp/ipykernel_489/689482210.py:8: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
/tmp/ipykernel_489/689482210.py:9: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
/tmp/ipykernel_489/689482210.py:39: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.


In [60]:
### Incorperate robert's radius prediction. Note: this is basing thins on FWI instead of ISI, and is temporarily ignooring the effect of the BUI. 
from math import e
from math import pi


### Implement BUI

def get_ros(df, isi_var, bui_var, a = 110, b = 0.0282, c = 1.5, spread_hours = 4, BUIo = 64, q = 0.70, lab = ""):
    df.loc[:, f"initial_rate_of_spread{lab}"] = a* (( 1 - (e**(-b * df[isi_var])))** c)
    df.loc[:, f"buildup_effect{lab}"] = e**(50 * (np.log(q)) * ((1/df[bui_var]) - (1/BUIo)))  #BuildupEffect = exp(50*log(q)*(1/BUIforROS - 1/BUIo))
    df.loc[:, f"rate_of_spread_eq{lab}"] = df[f"buildup_effect{lab}"] * df[f"initial_rate_of_spread{lab}"]
    df.loc[:, f"rate_of_spread{lab}"] = (df.loc[:, f"rate_of_spread_eq{lab}"]*(spread_hours*60)) / 1000
    return(df)
# def get_radius(df):
#     df.loc[:, "radius"] = (df.farea/pi)^(1/2)
#     return(df)

fire3 = get_ros(fire3, isi_var = "ISI", bui_var = "BUI")
fire3 = get_ros(fire3, isi_var = "ISI_forecast_average_shifted", bui_var = "BUI_forecast_average_shifted", lab = "_forecast_average_shifted") #FWI_forecast_average_shifted
fire3.loc[:, "radius"] = (fire3.farea_diff_stand/pi)**(1/2)


In [61]:
fire3 = fire3.sort_values(by = ["fireID", "t"])
fire3 = fire3[~fire3.BUI.isna()]
fire3 = fire3.groupby("fireID").apply(corrected_farea_diff).reset_index(drop = True) ## No supression signal for now. 

/tmp/ipykernel_489/999561434.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [62]:
def relative_growth(df):
    df["relative_growth"] =  (df.farea_diff + df.farea_shifted_back) / df.farea_shifted_back
    return(df)



In [63]:
fire3 = fire3.groupby("fireID").apply(relative_growth).reset_index(drop = True)

/tmp/ipykernel_489/4158884599.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


# Make Sampleing Scheme

I want to generate "rate parameters" that I can then test the skill of against out-of-sample fires. I want to make two-halves, each with a similar representation of large fires and small fires. 

Seems like "greedy matching" where you sort by and then create one and then the other, is pretty good. 


In [64]:
max_area = fire3.groupby("fireID").farea.max()
max_area = pd.DataFrame({"fireID": max_area.index, 
                        "farea_max" : max_area.values})


max_area = max_area.sort_values(by = "farea_max")
fit_ids = max_area.iloc[::2].fireID
test_ids = max_area.iloc[1::2].fireID



fire = fire3[fire3.fireID.isin(fit_ids)]
fire_test = fire3[fire3.fireID.isin(test_ids)]


# Trying a Bayesian model fit of Roberts model

In [65]:
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import pytensor.tensor as pt

In [66]:
table_6 = pd.DataFrame({
    'Fuel type': ['C-1', 'C-2', 'C-3', 'C-4', 'C-5', 'C-6', 'C-7', 'D-1', 'S-1', 'S-2', 'S-3', 'O-1a', 'O-1b'],
    'a': [90, 110, 110, 110, 30, 30, 45, 30, 75, 40, 55, 190, 250],
    'b': [0.0649, 0.0282, 0.0444, 0.0293, 0.0697, 0.0800, 0.0305, 0.0232, 0.0297, 0.0438, 0.0829, 0.0310, 0.0350],
    'c': [4.5, 1.5, 3.0, 1.5, 4.0, 3.0, 2.0, 1.6, 1.3, 1.7, 3.2, 1.4, 1.7]
})


table_7 = pd.DataFrame({
    'Fuel type': ['C-1', 'C-2', 'C-3', 'C-4', 'C-5', 'C-6', 'C-7', 'S-1', 'S-2', 'S-3'],
    'BUI0': [72, 64, 62, 66, 56, 62, 106,  38, 63, 31],
    'q': [0.90, 0.70, 0.75, 0.80, 0.80, 0.80, 0.85, 0.75, 0.75, 0.75],
    'Max. BE': [1.078, 1.321, 1.261, 1.184, 1.220, 1.197, 1.134, 1.460, 1.256, 1.590]
})


In [67]:
def log_normal_params(mu, var):
    alpha = np.log(mu) - ((1/2)*np.log((var + mu**2)/(mu**2))) ## assuming var is sigma squred, not signma in hooten and hobbs
    beta = (np.log((var + mu**2)/(mu**2)))**(1/2)
    return(alpha, beta)

def beta_dis_params(mu, var):
    alpha = (mu**2 - mu**3 - (mu**var))/var
    beta = (mu - (2*mu**2) + (mu**3) - var + (mu*var))/var
    return(alpha, beta)

# a_mu, a_sigma = log_normal_params(110, table_6.a.var())
# b_mu, b_sigma = beta_dis_params(0.0284, table_6.b.var())
# c_mu, c_sigma = log_normal_params(1.5, table_6.c.var())
# BUIo_mu, BUIo_sigma = log_normal_params(64, table_7.BUI0.var())
# q_mu, q_sigma = beta_dis_params(0.7, table_7.q.var())


def find_gamma_parameters(upper_bound=30, percentile=0.95):
    # We want to solve for mu and sigma where:
    # P(X ≤ 30) = 0.95
    # and X follows Gamma(α, β)
    
    def objective(mu_sigma):
        mu, sigma = mu_sigma
        alpha = (mu/sigma)**2
        beta = mu/sigma**2
        
        # Calculate the CDF at upper_bound
        cdf = stats.gamma.cdf(upper_bound, a=alpha, scale=1/beta)
        return abs(cdf - percentile)
    
    # Use optimization to find mu and sigma
    from scipy.optimize import minimize
    
    # Initial guess
    result = minimize(objective, x0=[15, 7], method='Nelder-Mead')
    
    mu, sigma = result.x
    alpha = (mu/sigma)**2
    beta = mu/sigma**2
    
    return mu, sigma


isi_mu, isi_sig = find_gamma_parameters(30, 0.95)
bui_mu, bui_sig = find_gamma_parameters(200, 0.95)

In [76]:
fire_delta = fire
fire_delta = fire.dropna()

def test_ros(df, isi_var = "ISI", bui_var = "BUI", a = 110, c = 1.5, BUIo = 64, q = 0.7, spread_hours = 4):
    # buildup_effect = np.exp(50 * np.log(q) * (1/df['BUI'] - 1/BUIo))
    # initial_ros = a * (1 - np.exp(-b * df['ISI']))**c
    # return (buildup_effect * initial_ros * (spread_hours*60)) / 1000
    ros = ( ((e**(50 * (np.log(q)) * ((1/df[bui_var]) - (1/BUIo)))) * (a* (( 1 - (e**(-b * df[isi_var])))** c)))*(spread_hours*60)) / 1000
    return(ros)


ros = test_ros(fire_delta)

ValueError: setting an array element with a sequence.

In [90]:
from math import e


with pm.Model() as model:

    # start = {
    # 'alpha_isi': 1.0,
    # 'alpha_bui': 1.0,
    # 'beta_isi': 1.0,
    # 'beta_bui': 1.0,
    # 'sigma': 1.0,
    # 'spread_hours': 4.0,
    # 'a': 110,
    # 'b': 0.0282,
    # 'c': 1.5,
    # 'BUIo': 64,
    # 'q': 0.7

    # }

    start = {
    'sigma': 1.0,
    'spread_hours': 4.0,
    'a': 110,
    'b': 0.0282,
    'c': 1.5,
    'BUIo': 64,
    'q': 0.7
    }
    # alpha_isi = pm.Uniform("alpha_isi", lower=1, upper=4)
    # alpha_bui = pm.Uniform("alpha_bui", lower=1, upper=4)

    # beta_isi = pm.Uniform("beta_isi", lower=1, upper=4)
    # beta_bui = pm.Uniform("beta_bui", lower=1, upper=4)
    sigma = pm.Uniform("sigma", lower = 1, upper = 2)


    spread_hours = pm.Uniform("spread_hours", lower = 0.5, upper = 12)
    #a = pm.LogNormal("a",  mu = a_mu, sigma = a_sigma) 
    #b = pm.Beta("b", alpha = b_mu, beta = b_sigma ) ## Change to uniform? or beta?
    #c = pm.LogNormal("c", mu = c_mu, sigma = c_sigma )
    #BUIo = pm.LogNormal("BUIo", mu = BUIo_mu, sigma = BUIo_sigma )
    #q = pm.Beta("q", alpha = q_mu, beta = q_sigma)## Change to unifrom? or beta 

    a = pm.Uniform("a",  lower = table_6.a.min(), upper = table_6.a.max()) 
    b = pm.Uniform("b",  lower = table_6.b.min(), upper = table_6.b.max()) 
    c = pm.Uniform("c",  lower = table_6.c.min(), upper = table_6.c.max()) 
    BUIo =  pm.Uniform("BUIo",  lower = 63, upper = 65) 
    q = pm.Uniform("q",  lower = table_7.q.min(), upper = table_7.q.max())  

    
    # isi_var = pm.Gamma("isi_var", alpha=alpha_isi, beta =beta_isi, observed=fire_delta.ISI)
    # bui_var = pm.Gamma("bui_var", alpha= alpha_bui, beta = beta_bui, observed=fire_delta.BUI)

    isi_var = pm.Gamma("isi_var", mu= isi_mu, sigma = isi_sig, observed=fire_delta.ISI)
    bui_var = pm.Gamma("bui_var", mu= bui_mu, sigma = bui_sig, observed=fire_delta.BUI)

    print(f"isi_mu: {isi_mu}, isi_sig: {isi_sig}")
    print(f"bui_mu: {bui_mu}, bui_sig: {bui_sig}")
    
        # Break down the calculation
    bui_ratio = pm.math.switch(bui_var > 0, 1/bui_var, 1e-10)  # Prevent division by zero
   
    bui_diff = bui_ratio - 1/BUIo
    print(f"{BUIo}")
    print( print(f"bui ratio: {bui_ratio}, bui_diff: {bui_diff}"))
    
    # Limit the exponential term
    exp_term = pm.math.clip(50 * pm.math.log(q) * bui_diff, -50, 50)
    buildup_effect = pm.math.exp(exp_term)
    
    # Calculate initial ROS with bounds
    exp_isi = pm.math.clip(-b * isi_var, -50, 50)
    initial_ros = a * (1 - pm.math.exp(exp_isi))**c
    
    # Combine terms
    ros_raw = (buildup_effect * initial_ros * (spread_hours*60)) / 1000
    
    # Ensure positive rate of spread
    ros_final = pm.math.switch(ros_raw > 0, ros_raw, 1e-10)
    #initial_ros = a* (( 1 - (e**(-b * df[isi_var])))** c)
   # buildup_effect = e**(50 * (np.log(q)) * ((1/df[bui_var]) - (1/BUIo)))  #BuildupEffect = exp(50*log(q)*(1/BUIforROS - 1/BUIo))
    #rate_of_spread_eq =
    #rate_of_spread = pm.Gamma("y", mu = ( ((e**(50 * (np.log(q)) * ((1/bui_var) - (1/BUIo)))) * (a* (( 1 - (e**(-b * isi_var)))** c)))*(spread_hours*60)) / 1000, sigma = 1, observed = fire_delta.radius)
    #rate_of_spread = pm.Gamma("y", mu = ( ((e**(50 * (np.log(q)) * ((1/bui_var) - (1/BUIo)))) * (a* (( 1 - (e**(-b * isi_var)))** c)))*(spread_hours*60)) / 1000, sigma = 1, observed = fire_delta.radius)

    # rate_of_spread = pm.Gamma("y", mu = ((pm.math.exp(50 * pm.math.log(q) * (1/bui_var - 1/BUIo))) * 
    #     (a * (1 - pm.math.exp(-b * isi_var))**c) * 
    #     (spread_hours*60)) / 1000, sigma = 1, observed = fire_delta.radius)

    idata = pm.sample(2000, 
                      tune=1000,
                      target_accept=0.95,
                      return_inferencedata=True, 
                      initvals = start)
    

isi_mu: 15.656220351614197, isi_sig: 7.658333829863839
bui_mu: 15.0, bui_sig: 7.0
BUIo
bui ratio: Switch.0, bui_diff: Sub.0
None


Initializing NUTS using jitter+adapt_diag...
/srv/conda/envs/notebook/lib/python3.12/site-packages/pytensor/tensor/elemwise.py:731: RuntimeWarning: invalid value encountered in log
/srv/conda/envs/notebook/lib/python3.12/site-packages/pytensor/tensor/rewriting/elemwise.py:1023: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
/srv/conda/envs/notebook/lib/python3.12/site-packages/pytensor/tensor/elemwise.py:731: RuntimeWarning: divide by zero encountered in log
/srv/conda/envs/notebook/lib/python3.12/site-packages/pytensor/tensor/elemwise.py:731: RuntimeWarning: invalid value encountered in impl (vectorized)
/srv/conda/envs/notebook/lib/python3.12/site-packages/pytensor/scalar/basic.py:1964: RuntimeWarning: invalid value encountered in scalar add


SamplingError: Initial evaluation of model at starting point failed!
Starting values:
{'sigma_interval__': array(-inf), 'spread_hours_interval__': array(-1.52703751), 'a_interval__': array(-0.19927143), 'b_interval__': array(-2.20025233), 'c_interval__': array(-2.40162147), 'BUIo_interval__': array(-0.45507023), 'q_interval__': array(-inf)}

Logp initial evaluation results:
{'sigma': np.float64(nan), 'spread_hours': np.float64(-1.92), 'a': np.float64(-1.4), 'b': np.float64(-2.41), 'c': np.float64(-2.58), 'BUIo': np.float64(-1.44), 'q': np.float64(nan), 'isi_var': np.float64(-29.72), 'bui_var': np.float64(-29.39)}
You can call `model.debug()` for more details.

In [84]:
model.debug(verbose = True)

/srv/conda/envs/notebook/lib/python3.12/site-packages/pytensor/tensor/elemwise.py:731: RuntimeWarning: invalid value encountered in log


point={'sigma_interval__': array(0.), 'spread_hours_interval__': array(0.), 'a_interval__': array(0.), 'b_interval__': array(0.), 'c_interval__': array(-1.66533454e-16), 'BUIo_interval__': array(nan), 'q_interval__': array(8.8817842e-16)}

The variable BUIo has the following parameters:
0: 63 [id A] <Scalar(int8, shape=())>
1: 65 [id B] <Scalar(int8, shape=())>
The parameters evaluate to:
0: 63
1: 65
Some of the values of variable BUIo are associated with a non-finite logp:
 value = nan -> logp = nan



In [ ]:
! pip install bambi

In [ ]:
import bambi as bmb

with pm.Model() as simple_model:
    sigma = pm.HalfCauchy("sigma", beta=10)
    intercept = pm.Normal("Intercept", 0, sigma=20)
    slope = pm.Normal("slope", 0, sigma=20)

    radius = pm.Normal("radius", mu = intercept + slope * fire.rate_of_spread, sigma = sigma, observed = fire.radius)
    idata = pm.sample(3000)

In [ ]:
isi_var = "ISI"
bui_var = "BUI"

data = fire[["radius", "rate_of_spread"]]
data = data.dropna()

bmb.Model("radius ~ rate_of_spread", data)